<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%209/1_st445_week9_lab1_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 9 · Lab: evaluating and communicating classifier performance

**About 2 hours · follows the Week 9 lecture section by section · 22 exercises**

The lecture used a synthetic fraud problem. In this lab you apply the same tools to a **real medical screening problem**: predicting diabetes from routine measurements. Each section:

1. explains the technique and why it is used, with links to the documentation;
2. runs a short **demonstration** and interprets its output;
3. sets **two exercises**: 🟢 a simple one that practises the demonstrated tool, and 🔴 a complex one that extends it or asks you to justify a decision.

| Section | Topic (lecture section) |
|---|---|
| 0 | Data and set-up |
| 1 | Why one number is not enough (01) |
| 2 | Confusion matrices (02) |
| 3 | ROC curves and AUC (03) |
| 4 | Precision–recall curves (04) |
| 5 | Calibration (05) |
| 6 | Learning and validation curves (06) |
| 7 | Slice-based error analysis (07) |
| 8 | Beyond classification: regression and clustering (08) |
| 9 | Choosing a threshold (09) |
| 10 | Pitfalls: leakage and uncertainty (11) |
| 11 | Communicating performance (12) |

**Main documentation:** scikit-learn [User Guide: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html), [Visualizations (the Display API)](https://scikit-learn.org/stable/visualizations.html), [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html), [Tuning the decision threshold](https://scikit-learn.org/stable/modules/classification_threshold.html).

**Before you start:** choose **File → Save a copy in Drive**.

---
## 0. The data: diabetes screening

The **Pima Indians Diabetes** dataset records eight routine measurements for 768 women aged 21 or over, of Pima heritage, near Phoenix, Arizona, and whether they were diagnosed with diabetes within five years (Smith et al., 1988). It is a classic screening problem: about a third of the women developed diabetes, the measurements overlap between the groups, and the two kinds of error have very different consequences for a patient.

* **Source:** originally the US National Institute of Diabetes and Digestive and Kidney Diseases; Smith, J. W. et al. (1988), "Using the ADAP learning algorithm to forecast the onset of diabetes mellitus", *Proceedings of the Symposium on Computer Applications in Medical Care*, 261–265. We load a widely used copy from [GitHub](https://github.com/jbrownlee/Datasets).
* **Columns:** pregnancies; plasma **glucose** (2-hour oral glucose tolerance test, mg/dL); diastolic **blood pressure** (mm Hg); triceps **skin thickness** (mm); 2-hour serum **insulin** (µU/mL); **BMI** (kg/m²); diabetes **pedigree** function (family history score); **age** (years); outcome (1 = diabetes).

### Imports and style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     LearningCurveDisplay, ValidationCurveDisplay, TunedThresholdClassifierCV)
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, roc_curve, precision_recall_curve, make_scorer,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,       # remove the top border of each plot
    "axes.spines.right": False,     # remove the right border of each plot
    "axes.prop_cycle": plt.cycler(color=OKABE_ITO),   # colour-blind-safe colours by default
})
RNG = 445                           # one seed for every random step

### Loading and preprocessing, in one cell

One preprocessing decision matters here: in five columns a value of **0 is physiologically impossible** (nobody has a glucose, blood pressure or BMI of zero), so these zeros are **disguised missing values** (Week 4). We convert them to `NaN`. We do **not** impute them here: imputation is done inside each model's pipeline, so it is fitted on training data only (Section 10 shows why).

In [ ]:
# 🟢 Load and prepare the data: one documented cell
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLUMNS = ["pregnancies", "glucose", "blood_pressure", "skin_thickness", "insulin", "bmi", "pedigree", "age", "diabetes"]

raw = pd.read_csv(URL, header=None, names=COLUMNS)    # the file has no header row
pima = raw.copy()                                      # keep raw unchanged

# 1. Zeros that cannot be real measurements are missing values
IMPOSSIBLE_ZERO = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
pima[IMPOSSIBLE_ZERO] = pima[IMPOSSIBLE_ZERO].replace(0, np.nan)

# 2. Features and outcome
X = pima.drop(columns="diabetes")
y = pima["diabetes"]

# 3. One split for the whole lab, stratified so both halves have the same share of diabetes.
#    The test set is used ONLY to evaluate: every choice is made on the training data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RNG)

assert len(pima) == 768 and y.isin([0, 1]).all()
print("women:", len(pima), "| diabetes:", int(y.sum()), f"({y.mean():.1%})")
print("missing values after recoding:", pima.isna().sum()[pima.isna().sum() > 0].to_dict())
print("training:", X_train.shape, "| test:", X_test.shape, f"| test prevalence {y_test.mean():.1%}")

**What you should see:** 768 women, 268 with diabetes (34.9%). Insulin is missing for 374 women and skin thickness for 227, so a model that cannot handle missing values needs imputation. The test set has 231 women, 81 with diabetes.

### Three models, each a pipeline

A [`pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) chains preprocessing and a model, so that `fit` fits the imputer and the scaler on the training data only, and `predict_proba` applies them to new data. We use the three models of the lecture's comparison:

* **logistic regression** ([`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)), with median imputation ([`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)) and standardisation ([`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html));
* **random forest** ([`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html));
* **naive Bayes** ([`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)).

`P` stores each model's predicted probability of diabetes for the 231 test women: the input to every plot that follows.

In [ ]:
models = {
    "logistic regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=2000)),
    "random forest": make_pipeline(SimpleImputer(strategy="median"),
                                   RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RNG)),
    "naive Bayes": make_pipeline(SimpleImputer(strategy="median"), GaussianNB()),
}
P = {}
for name, model in models.items():
    model.fit(X_train, y_train)                         # imputer, scaler and model fitted on training data only
    P[name] = model.predict_proba(X_test)[:, 1]         # probability of diabetes for each test woman
print({name: p[:5].round(2) for name, p in P.items()})

---
## 1. Why one number is not enough

A single metric answers a single question. We compare the three models on several metrics at the default threshold of 0.5, beside a **baseline** that always predicts "no diabetes" ([`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) with `strategy="most_frequent"`). A model is only useful if it beats such a baseline on the metric that matters.

Metrics used: [accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) (share of women with diabetes who are flagged), [precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) (share of flagged women who have diabetes), [ROC AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html) and the [Brier score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html).

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

rows = {"always 'no diabetes'": {"accuracy": accuracy_score(y_test, baseline.predict(X_test)), "recall": 0.0}}
for name, p in P.items():
    y_hat = (p >= 0.5).astype(int)                      # labels at the default threshold
    rows[name] = {"accuracy": accuracy_score(y_test, y_hat), "recall": recall_score(y_test, y_hat),
                  "precision": precision_score(y_test, y_hat), "ROC AUC": roc_auc_score(y_test, p),
                  "average precision": average_precision_score(y_test, p), "Brier score": brier_score_loss(y_test, p)}
metrics_table = pd.DataFrame(rows).T.round(3)
metrics_table

**Interpreting the output:** Doing nothing is 64.9% accurate, because 65% of the test women do not have diabetes. Logistic regression reaches 76.6% accuracy but flags only 53% of the women who have diabetes (recall 0.531). Naive Bayes is less accurate (75.3%) but catches more cases (recall 0.605); logistic regression has the best ROC AUC (0.832), average precision (0.718) and Brier score (0.157). **Which model is 'best' depends on the metric**, and for screening, where a missed case is costly, the high accuracy of a model that misses half the cases is not reassuring.

#### Exercise 1.1 · 🟢 Simple

Accuracy treats both classes as one pool. Compute the **balanced accuracy** ([`balanced_accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html): the mean of recall and specificity) of the baseline and of the three models at the threshold 0.5. How does the baseline's score change compared with its accuracy?

In [ ]:
# Exercise 1.1: your code here

✍️ *Your interpretation:*

#### Exercise 1.2 · 🔴 Complex

Build a table that **ranks** the three models (1 = best) under each of these metrics: accuracy, balanced accuracy, recall, precision, F1 ([`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)), Matthews correlation coefficient ([`matthews_corrcoef`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.matthews_corrcoef.html)), ROC AUC, average precision and Brier score (lower is better). Which metrics change the ranking? Write two sentences explaining which metric you would report to a clinician deciding whom to send for a confirmatory test, and why.

In [ ]:
# Exercise 1.2: your code here

✍️ *Your interpretation:*

---
## 2. Confusion matrices

A confusion matrix counts the four outcomes at one threshold: true negatives, false positives, false negatives and true positives. [`ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) draws it; `normalize="true"` divides each **row** by its true-class total (so the diagonal shows recall and specificity), and `normalize="pred"` divides each **column** by its predicted total (so the diagonal shows precision for each prediction). In scikit-learn, rows are the true class and columns the prediction ([User Guide: confusion matrix](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix)).

In [ ]:
y_hat = (P["logistic regression"] >= 0.5).astype(int)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3), layout="constrained")
for ax, norm, title, fmt in [(axs[0], None, "Counts", "d"),
                             (axs[1], "true", "Row-normalised (by true class)", ".2f"),
                             (axs[2], "pred", "Column-normalised (by prediction)", ".2f")]:
    ConfusionMatrixDisplay.from_predictions(y_test, y_hat, normalize=norm, values_format=fmt,
                                            display_labels=["no diabetes", "diabetes"],
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(title)
plt.show()
print("TN, FP, FN, TP =", confusion_matrix(y_test, y_hat).ravel())

**Interpreting the output:** Of 231 test women, 134 without diabetes are correctly cleared, 16 are flagged unnecessarily, 38 with diabetes are missed and 43 are correctly flagged. The **row-normalised** matrix shows the asymmetry: 89% of women without diabetes are correctly cleared, but only 53% of women with diabetes are flagged; 47% are missed. The **column-normalised** matrix shows that 73% of flagged women do have diabetes (precision), and 78% of cleared women are truly clear. The default threshold favours avoiding false alarms at the price of many missed cases.

#### Exercise 2.1 · 🟢 Simple

From the four counts printed above, compute **by hand** (with arithmetic on `tn, fp, fn, tp`) the recall, specificity, precision and accuracy of logistic regression at 0.5. Check each against the corresponding scikit-learn function or the normalised matrices.

In [ ]:
# Exercise 2.1: your code here

✍️ *Your interpretation:*

#### Exercise 2.2 · 🔴 Complex

Draw the **row-normalised** confusion matrix of logistic regression at three thresholds, **0.3, 0.5 and 0.7**, side by side. Describe how the two kinds of error move as the threshold changes, and say which threshold you would recommend for a screening programme where a confirmatory test follows a positive result.

In [ ]:
# Exercise 2.2: your code here

✍️ *Your interpretation:*

---
## 3. ROC curves and AUC

An ROC curve plots the true positive rate against the false positive rate at **every** threshold ([`roc_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)). The area under it ([`roc_auc_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)) is the probability that a randomly chosen woman with diabetes gets a higher score than a randomly chosen woman without (Hanley and McNeil, 1982). [`RocCurveDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html) draws the curve; `plot_chance_level=True` adds the diagonal of random ranking ([User Guide: ROC](https://scikit-learn.org/stable/modules/model_evaluation.html#receiver-operating-characteristic-roc)).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for i, (name, model) in enumerate(models.items()):
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax,
                                   plot_chance_level=(i == len(models) - 1))   # draw the diagonal once
ax.set(title="ROC curves, diabetes test set (n = 231)")
ax.set_aspect("equal")
plt.show()

**Interpreting the output:** The three curves are close and cross several times: logistic regression has the largest area (AUC 0.832), random forest 0.821 and naive Bayes 0.798, but no model is above the others at every threshold. With only 81 women with diabetes in the test set, each curve moves in steps of 1/81 in TPR, and differences of 0.01 in AUC are well within sampling noise (Exercise 10.2 measures this).

#### Exercise 3.1 · 🟢 Simple

Compute each model's ROC AUC and its **partial AUC** for false positive rates up to 0.2, with `roc_auc_score(..., max_fpr=0.2)`. Does the ranking of the models change in the low-FPR region, which matters when false alarms must be few?

In [ ]:
# Exercise 3.1: your code here

✍️ *Your interpretation:*

#### Exercise 3.2 · 🔴 Complex

Build the ROC curve of logistic regression **by hand**: for each distinct predicted probability t (from highest to lowest), compute TPR(t) and FPR(t) with NumPy, plot the curve, and compute its area with the trapezium rule (`np.trapezoid`). Then verify the ranking interpretation: compute the share of (diabetes, no diabetes) pairs in which the woman with diabetes has the higher score (ties count half). All three numbers should agree with `roc_auc_score`.

In [ ]:
# Exercise 3.2: your code here

✍️ *Your interpretation:*

---
## 4. Precision–recall (PR) curves

A PR curve plots precision against recall at every threshold ([`precision_recall_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html)); [`average_precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html) summarises it as a step-wise area. The chance level is a horizontal line at the **prevalence**, not a diagonal. [`PrecisionRecallDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.PrecisionRecallDisplay.html) draws it ([User Guide: precision–recall](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics)).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for i, (name, model) in enumerate(models.items()):
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax,
                                          plot_chance_level=(i == len(models) - 1))   # the prevalence line, once
ax.set(title=f"Precision–recall curves (prevalence {y_test.mean():.1%})", ylim=(0, 1.02))
plt.show()

**Interpreting the output:** At low recall all three models are precise (the women with the highest scores almost all have diabetes); precision falls as the threshold is lowered to catch more cases. Average precision is 0.718 for logistic regression, 0.697 for random forest and 0.671 for naive Bayes, against a chance level of 0.351, the test prevalence. To catch 80% of cases, about half of the flagged women would not have diabetes (Exercise 4.1 quantifies this).

#### Exercise 4.1 · 🟢 Simple

For each model, find the **highest precision achievable while keeping recall at least 0.8**, using the arrays returned by `precision_recall_curve`. Report it with the threshold that achieves it.

In [ ]:
# Exercise 4.1: your code here

✍️ *Your interpretation:*

#### Exercise 4.2 · 🔴 Complex

Show that **AP depends on prevalence but ROC AUC does not**. Using logistic regression's test scores, draw 200 random subsamples in which diabetes makes up only **10%** of the women (keep all women without diabetes, sample the right number with diabetes). For each, compute ROC AUC and AP. Compare the mean and standard deviation with the full test set, and plot the two distributions. Explain the result in two sentences.

In [ ]:
# Exercise 4.2: your code here

✍️ *Your interpretation:*

---
## 5. Calibration

A model is **calibrated** if, among women given a probability of about 0.3, about 30% develop diabetes. [`CalibrationDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html) draws a **reliability diagram**: the mean predicted probability in each bin against the observed fraction with diabetes; the diagonal is perfect calibration. The Brier score is the mean squared error of the probabilities: lower is better ([User Guide: calibration](https://scikit-learn.org/stable/modules/calibration.html); Niculescu-Mizil and Caruana, 2005).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for name, model in models.items():
    CalibrationDisplay.from_estimator(model, X_test, y_test, n_bins=5, name=f"{name} (Brier {brier_score_loss(y_test, P[name]):.3f})", ax=ax)
ax.set(title="Reliability diagram, 5 bins (n = 231)")
plt.show()

**Interpreting the output:** Logistic regression lies close to the diagonal in every bin (Brier 0.157). Naive Bayes is the least calibrated (Brier 0.191): its probabilities are too extreme, with predictions near 0.5 coming true only about a quarter of the time and very low predictions too low. Random forest is close in the low bins but uneven at the top. With only 231 women and 5 bins, each point rests on a few dozen women, so small departures are noise.

#### Exercise 5.1 · 🟢 Simple

For each model, compare the **mean predicted probability** on the test set with the **observed rate** of diabetes (`y_test.mean()`), and report the Brier score. Which model's probabilities are, on average, too high or too low?

In [ ]:
# Exercise 5.1: your code here

✍️ *Your interpretation:*

#### Exercise 5.2 · 🔴 Complex

Recalibrate **naive Bayes** with [`CalibratedClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html), using `method="isotonic"` and, separately, `method="sigmoid"` (Platt scaling), each with `cv=5` on the training data. Compare ROC AUC and Brier score before and after, and draw the reliability diagrams of the original and the two recalibrated models. Explain why AUC barely changes while the Brier score improves.

In [ ]:
# Exercise 5.2: your code here

✍️ *Your interpretation:*

---
## 6. Learning and validation curves

[`LearningCurveDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LearningCurveDisplay.html) plots training and cross-validated scores as the training set grows: a large, closing gap means high variance (more data would help); two low curves that meet early mean high bias. [`ValidationCurveDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ValidationCurveDisplay.html) does the same against one hyperparameter ([User Guide: validation curves](https://scikit-learn.org/stable/modules/learning_curve.html)).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
fig, ax = plt.subplots(figsize=(7, 4.5))
LearningCurveDisplay.from_estimator(models["logistic regression"], X_train, y_train, cv=cv, scoring="roc_auc",
                                    train_sizes=np.linspace(0.1, 1.0, 8), score_type="both", ax=ax)
ax.set(title="Learning curve: logistic regression (ROC AUC, 5-fold CV)")
plt.show()

**Interpreting the output:** The training and validation curves of logistic regression stay close (training about 0.85, validation about 0.83 at full size): a small gap, so the model is not overfitting, and the validation curve rises only slowly (from 0.79 to 0.83), so more data would help little. This is a **bias-limited** model: gains would come from better features or a more flexible model, not from more women. The shaded band is the spread across the 5 folds.

#### Exercise 6.1 · 🟢 Simple

Draw a **validation curve** for logistic regression's regularisation strength `C` over `np.logspace(-3, 2, 11)` (`param_name="logisticregression__C"`, because the model is inside a pipeline). Which value of C gives the best cross-validated AUC, and is there a region of underfitting?

In [ ]:
# Exercise 6.1: your code here

✍️ *Your interpretation:*

#### Exercise 6.2 · 🔴 Complex

Draw the learning curves of **random forest** and **logistic regression** side by side (shared y-axis). Diagnose each model in terms of bias and variance, and explain which would benefit more from collecting data on more women.

In [ ]:
# Exercise 6.2: your code here

✍️ *Your interpretation:*

---
## 7. Slice-based error analysis

An overall metric averages over groups. We split the test women into **age groups** and compute recall per group, with the number of women with diabetes in each, because small groups give unreliable rates (Mitchell et al., 2019, model cards; Barocas, Hardt and Narayanan, 2023). [`pd.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) makes the groups.

In [ ]:
age_group = pd.cut(X_test["age"], bins=[20, 30, 45, 90], labels=["21–30", "31–45", "46+"])
y_hat = (P["logistic regression"] >= 0.5).astype(int)
slices = (pd.DataFrame({"group": age_group, "y": y_test.to_numpy(), "y_hat": y_hat})
          .groupby("group", observed=True)
          .apply(lambda d: pd.Series({"women": len(d), "with diabetes": int(d.y.sum()),
                                      "recall": recall_score(d.y, d.y_hat), "prevalence": d.y.mean()}), include_groups=False))
print(f"overall recall: {recall_score(y_test, y_hat):.3f}")
slices.round(3)

**Interpreting the output:** Overall recall is 0.531, but it differs by age: the youngest group has the lowest prevalence and the lowest recall, so young women with diabetes are the most likely to be missed. Each group's recall rests on only a few dozen cases, so the differences need intervals before anyone acts on them (Exercise 7.2).

#### Exercise 7.1 · 🟢 Simple

Compute the **false positive rate** (share of women without diabetes who are flagged) for each age group. Which group bears most of the unnecessary referrals?

In [ ]:
# Exercise 7.1: your code here

✍️ *Your interpretation:*

#### Exercise 7.2 · 🔴 Complex

Compute a **95% bootstrap interval** for recall in each age group (1,000 resamples of the women with diabetes in each group), and draw the result as a dot-and-interval plot with the number of cases beside each point and a line at the overall recall. Do the intervals support a claim that the model is less sensitive for younger women? Relate your answer to the **equal opportunity** criterion (Hardt et al., 2016).

In [ ]:
# Exercise 7.2: your code here

✍️ *Your interpretation:*

---
## 8. Beyond classification: regression and clustering

The same logic, plots first, applies to other models. For **regression**, [`PredictionErrorDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.PredictionErrorDisplay.html) draws actual against predicted values and residuals against predicted values; a pattern in the residuals means the model has missed structure. We use scikit-learn's bundled diabetes **progression** data ([`load_diabetes`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html): a disease-progression score one year after baseline for 442 patients) so the regression has a real numeric target.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import PredictionErrorDisplay, mean_absolute_error, mean_squared_error, r2_score
Xr, yr = load_diabetes(return_X_y=True, as_frame=True)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, random_state=RNG)
reg = LinearRegression().fit(Xr_tr, yr_tr)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
PredictionErrorDisplay.from_estimator(reg, Xr_te, yr_te, kind="actual_vs_predicted", ax=axs[0], scatter_kwargs={"alpha": 0.6})
PredictionErrorDisplay.from_estimator(reg, Xr_te, yr_te, kind="residual_vs_predicted", ax=axs[1], scatter_kwargs={"alpha": 0.6})
axs[0].set_title("Actual vs predicted"); axs[1].set_title("Residuals vs predicted")
plt.show()

**Interpreting the output:** Points scatter widely around the diagonal: the linear model captures the trend but with large errors. The residual plot shows no strong curve, so the linear form is not badly wrong; look also at whether the vertical spread changes with the predicted value (a funnel), which would mean the errors are larger for some patients than others.

#### Exercise 8.1 · 🟢 Simple

Compute the **MAE**, **RMSE** and **R²** of the linear regression on the test set ([`mean_absolute_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), [`mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [`r2_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)). Express the MAE in words for a clinician, using the range of the target.

In [ ]:
# Exercise 8.1: your code here

✍️ *Your interpretation:*

#### Exercise 8.2 · 🔴 Complex

Cluster the **standardised** Pima features (impute medians first) with [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) for k = 2 to 6. For each k compute the mean silhouette ([`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)), and draw the silhouette plot ([`silhouette_samples`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_samples.html)) for the best k. Do the clusters correspond to diabetes status (cross-tabulate)? What does this say about using clustering to find 'patient types'?

In [ ]:
# Exercise 8.2: your code here

✍️ *Your interpretation:*

---
## 9. Choosing a threshold from costs

Suppose a positive screening result leads to a confirmatory test costing **£100**, and a missed case of diabetes costs **£1,000** in later complications and treatment (illustrative costs; state such assumptions explicitly). For calibrated probabilities, the cost-minimising rule is to flag a woman when $$p > t* = \frac{C_{FP}}{(C_{FP}+C_{FN})} \text{(Elkan, 2001).}$$ We plot the expected cost per 1,000 women against the threshold.

In [ ]:
C_FP, C_FN = 100, 1000                                   # £ per false positive, £ per false negative
p = P["logistic regression"]; yt = y_test.to_numpy()

def cost_per_1000(t, p=p, yt=yt):
    flagged = p >= t
    fp = np.sum(flagged & (yt == 0)); fn = np.sum(~flagged & (yt == 1))
    return 1000 * (C_FP * fp + C_FN * fn) / len(yt)

thresholds = np.linspace(0.01, 0.9, 200)
costs = [cost_per_1000(t) for t in thresholds]
t_star = C_FP / (C_FP + C_FN)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, costs, lw=2)
ax.axvline(t_star, ls=":", color="grey"); ax.axvline(0.5, ls="--", color="grey")
ax.text(t_star + 0.01, max(costs) * 0.95, f"t* = {t_star:.3f}"); ax.text(0.51, max(costs) * 0.95, "default 0.5")
ax.set(xlabel="threshold", ylabel="expected cost per 1,000 women (£)", title="Expected cost against threshold, logistic regression")
plt.show()
print(f"cost per 1,000 at 0.5: £{cost_per_1000(0.5):,.0f} | at t*: £{cost_per_1000(t_star):,.0f} | lowest on test: £{min(costs):,.0f} at t = {thresholds[np.argmin(costs)]:.3f}")

**Interpreting the output:** With a missed case ten times as costly as a confirmatory test, t* = 100 / 1,100 ≈ 0.091. The default threshold of 0.5 is far too high: it costs much more per 1,000 women than thresholds near t*, because it misses about half the cases. The curve is flat around its minimum, so any threshold in that region is nearly as good; the lowest point on the test set is itself noisy, and choosing it there would be tuning on the test set (Section 10).

#### Exercise 9.1 · 🟢 Simple

Compute t* for two other cost settings: (a) confirmatory test £50, missed case £2,000; (b) confirmatory test £300, missed case £600. For setting (b), draw the row-normalised confusion matrix at t* and describe the trade-off.

In [ ]:
# Exercise 9.1: your code here

✍️ *Your interpretation:*

#### Exercise 9.2 · 🔴 Complex

Choose the threshold **without looking at the test set**, using [`TunedThresholdClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TunedThresholdClassifierCV.html) with a custom cost scorer ([`make_scorer`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html) with `greater_is_better=False`) and 5-fold CV on the training data. Report the chosen threshold, and the cost per 1,000 test women at that threshold, at t*, and at 0.5. Explain why the CV-chosen threshold is the honest one to report.

In [ ]:
# Exercise 9.2: your code here

✍️ *Your interpretation:*

---
## 10. Pitfalls: leakage and uncertainty

**Leakage** makes evaluation optimistic. A classic form is selecting features using all the data before cross-validation. To make it visible, we add 1,000 columns of pure **random noise** to the training data, select the 10 'best' columns with [`SelectKBest`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html) on **all** the training rows, and only then cross-validate (Kaufman et al., 2012; Kapoor and Narayanan, 2023).

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
rng = np.random.default_rng(RNG)
noise = rng.normal(size=(len(y_train), 1000))             # 1,000 features with no relationship to diabetes

selected = SelectKBest(f_classif, k=10).fit_transform(noise, y_train)    # WRONG: selection sees every row's label
leaky = cross_val_score(LogisticRegression(max_iter=2000), selected, y_train, cv=cv, scoring="roc_auc")
print(f"cross-validated AUC on pure noise, selection BEFORE CV: {leaky.mean():.3f} ± {leaky.std():.3f}")

**Interpreting the output:** The data are pure noise, so the true AUC is 0.5, yet cross-validation reports an AUC well above 0.5. The 10 columns were chosen because, by chance, they correlate with the labels of **all** the rows, including the rows later used for validation: information leaked from validation into training.

#### Exercise 10.1 · 🟢 Simple

Fix the leak: put `SelectKBest` **inside** a pipeline with the logistic regression, so that selection is refitted within each training fold, and cross-validate on the noise again. What AUC do you get now, and why?

In [ ]:
# Exercise 10.1: your code here

✍️ *Your interpretation:*

#### Exercise 10.2 · 🔴 Complex

Is logistic regression really better than random forest on this test set? Compute a **paired bootstrap** 95% interval for the difference in ROC AUC (1,000 resamples of the 231 test women; both models scored on each resample). Plot the distribution of the difference and state a conclusion suitable for a manager.

In [ ]:
# Exercise 10.2: your code here

✍️ *Your interpretation:*

---
## 11. Communicating performance

Non-specialists understand **natural frequencies** (counts out of a fixed reference group) far better than conditional probabilities (Gigerenzer and Hoffrage, 1995). We translate the screening rule into **outcomes per 1,000 women**, at the default threshold and at t*. The test-set rates are scaled to 1,000 women with the test prevalence.

In [ ]:
def per_1000(t, p=P["logistic regression"], yt=y_test.to_numpy()):
    """Outcomes per 1,000 women screened at threshold t, from the test set."""
    flagged = p >= t
    scale = 1000 / len(yt)
    return {"with diabetes": round(scale * (yt == 1).sum()),
            "correctly flagged": round(scale * np.sum(flagged & (yt == 1))),
            "missed": round(scale * np.sum(~flagged & (yt == 1))),
            "unnecessary confirmatory tests": round(scale * np.sum(flagged & (yt == 0))),
            "cost (£)": round(cost_per_1000(t))}

summary_1000 = pd.DataFrame({"default threshold 0.5": per_1000(0.5), f"cost-based threshold {t_star:.3f}": per_1000(t_star)})
summary_1000

**Interpreting the output:** Out of every 1,000 women screened, about 351 have diabetes. At the default threshold the programme finds about 186 of them and misses about 165, with about 69 unnecessary confirmatory tests. At the cost-based threshold it finds far more and misses far fewer, at the price of several hundred unnecessary tests, and its expected cost is much lower. This table, not an AUC, is what a health manager needs.

#### Exercise 11.1 · 🟢 Simple

Write a function that turns the per-1,000 numbers for a threshold into **one plain-English sentence** for a patient leaflet, for example: *'Of every 1,000 women screened, about X have diabetes; the test finds Y of them and misses Z, and W women without diabetes are asked to take a further test.'* Print it for t*.

In [ ]:
# Exercise 11.1: your code here

✍️ *Your interpretation:*

#### Exercise 11.2 · 🔴 Complex

Draw an **icon array** of 1,000 women for the cost-based threshold: one small square per woman, coloured by outcome (correctly flagged, missed, unnecessary further test, correctly cleared), with a legend giving the counts and a title that states the finding (Galesic et al., 2009). Use the colour-blind-safe palette. Then write one sentence on why this display suits a non-specialist better than a PR curve.

In [ ]:
# Exercise 11.2: your code here

✍️ *Your interpretation:*

---
## Summary

| Question | Tool | Section |
|---|---|---|
| Is the model better than doing nothing? | a baseline (`DummyClassifier`), several metrics | 1 |
| What errors does it make at one threshold? | confusion matrix, row- and column-normalised | 2 |
| How well does it rank cases? | ROC curve, AUC, partial AUC | 3 |
| How pure are its alerts at this prevalence? | PR curve, average precision | 4 |
| Can its probabilities be shown to patients? | reliability diagram, Brier score, recalibration | 5 |
| Would more data help? | learning and validation curves | 6 |
| Does it fail a group? | slice metrics with bootstrap intervals | 7 |
| Which threshold? | expected cost, t* = C_FP / (C_FP + C_FN), `TunedThresholdClassifierCV` | 9 |
| Is the evaluation honest? | pipelines; paired bootstrap of differences | 10 |
| How do we explain it? | outcomes per 1,000, a plain sentence, an icon array | 11 |

### References

Barocas, S., Hardt, M. and Narayanan, A. (2023), *Fairness and Machine Learning*, MIT Press · Elkan, C. (2001), "The foundations of cost-sensitive learning", *Proc. IJCAI* · Galesic, M., Garcia-Retamero, R. and Gigerenzer, G. (2009), *Health Psychology*, 28(2), 210–216 · Gigerenzer, G. and Hoffrage, U. (1995), *Psychological Review*, 102(4), 684–704 · Hanley, J. A. and McNeil, B. J. (1982), *Radiology*, 143(1), 29–36 · Hardt, M., Price, E. and Srebro, N. (2016), "Equality of opportunity in supervised learning", *NeurIPS* · Kapoor, S. and Narayanan, A. (2023), *Patterns*, 4(9), 100804 · Kaufman, S. et al. (2012), *ACM TKDD*, 6(4), 15 · Mitchell, M. et al. (2019), "Model cards for model reporting", *Proc. FAT\** · Niculescu-Mizil, A. and Caruana, R. (2005), *Proc. ICML* · Pedregosa, F. et al. (2011), "Scikit-learn", *JMLR*, 12, 2825–2830 · Smith, J. W. et al. (1988), *Proc. Symposium on Computer Applications in Medical Care*, 261–265.